# CreditLens — Day 4 Random Forest review

This executed notebook reviews saved Day 4 artefacts. It does not retrain, rescore validation, load final-test outcomes, select a threshold or perform SHAP analysis.

In [1]:
from pathlib import Path
import csv
import json

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'reports').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
metrics = json.loads((PROJECT_ROOT / 'reports/day4_random_forest_validation_metrics.json').read_text())
selected = json.loads((PROJECT_ROOT / 'reports/day4_random_forest_selected_params.json').read_text())
logistic = json.loads((PROJECT_ROOT / 'reports/day3_logistic_validation_metrics.json').read_text())

## Training-only grouped search

In [2]:
print(f"Configurations: {selected['search_iterations']}")
print(f"CV fits: {selected['fitted_candidates']}")
print(f"Search seconds: {selected['search_seconds']:.3f}")
print(f"Maximum groups overlapping in any fold: {max(f['overlapping_groups'] for f in selected['cv_group_audit'])}")
selected['selected_hyperparameters']

Configurations: 12
CV fits: 48
Search seconds: 1144.966
Maximum groups overlapping in any fold: 0


{'class_weight': None,
 'max_depth': 20,
 'max_features': 'sqrt',
 'min_samples_leaf': 10,
 'min_samples_split': 2,
 'n_estimators': 300}

## Validation comparison

Average Precision (AP) is calculated using `average_precision_score`. Threshold 0.5 is for reporting only. Small differences on one validation partition do not prove superiority.

In [3]:
fields = ['roc_auc', 'average_precision', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5', 'proportion_flagged_at_0_5']
{field: {'Logistic Regression': logistic[field], 'Random Forest': metrics[field]} for field in fields}

{'roc_auc': {'Logistic Regression': 0.7544054889873056,
  'Random Forest': 0.77191665762253},
 'average_precision': {'Logistic Regression': 0.519470900444617,
  'Random Forest': 0.539979526640148},
 'precision_at_0_5': {'Logistic Regression': 0.6696165191740413,
  'Random Forest': 0.6618287373004355},
 'recall_at_0_5': {'Logistic Regression': 0.3421250941974378,
  'Random Forest': 0.3436322532027129},
 'f1_at_0_5': {'Logistic Regression': 0.45286783042394013,
  'Random Forest': 0.4523809523809524},
 'proportion_flagged_at_0_5': {'Logistic Regression': 0.113,
  'Random Forest': 0.11483333333333333}}

## Random Forest validation curves

![ROC](../reports/figures/day4_random_forest_validation_roc.png)

![Precision–recall](../reports/figures/day4_random_forest_validation_precision_recall.png)

## Importance artefacts

Impurity importance can favour continuous or high-cardinality inputs. Correlated predictors can split, mask or inflate both impurity and permutation importance. These values are not causal.

In [4]:
with (PROJECT_ROOT / 'reports/day4_random_forest_source_importance.csv').open(newline='') as source:
    source_importance = list(csv.DictReader(source))
source_importance[:10]

[{'source_feature': 'PAY_0',
  'aggregated_impurity_importance': '0.223262424637',
  'permutation_ap_mean': '0.163763909399',
  'permutation_ap_std': '0.00789585164479'},
 {'source_feature': 'PAY_2',
  'aggregated_impurity_importance': '0.107516571326',
  'permutation_ap_mean': '0.0193288484489',
  'permutation_ap_std': '0.00452113464889'},
 {'source_feature': 'PAY_AMT1',
  'aggregated_impurity_importance': '0.0464855647072',
  'permutation_ap_mean': '0.010485099893',
  'permutation_ap_std': '0.00130783340384'},
 {'source_feature': 'PAY_3',
  'aggregated_impurity_importance': '0.0674345898397',
  'permutation_ap_mean': '0.00935707709174',
  'permutation_ap_std': '0.00184523282355'},
 {'source_feature': 'LIMIT_BAL',
  'aggregated_impurity_importance': '0.0463733100576',
  'permutation_ap_mean': '0.00914216921781',
  'permutation_ap_std': '0.00248608636672'},
 {'source_feature': 'PAY_4',
  'aggregated_impurity_importance': '0.0608865740986',
  'permutation_ap_mean': '0.00636845536968',
 

In [5]:
{
    'validation_used_for_selection': metrics['validation_used_for_hyperparameter_selection'],
    'test_partition_evaluated': metrics['test_partition_evaluated'],
    'threshold_status': metrics['threshold_status'],
    'search_warnings': metrics['search_warnings'],
    'resource_notes': metrics.get('resource_notes', []),
}

{'validation_used_for_selection': False,
 'test_partition_evaluated': False,
 'threshold_status': 'Baseline reporting only; not an operating threshold.',
 'search_warnings': [],
 'resource_notes': ['Initial sparse-input search attempt was interrupted for excessive runtime before validation.',
  'Completed dense-input search was retained; post-search permutation importance resumed serially after sandbox process creation was denied.']}